In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import sys


sys.path.append(Path("..").resolve().as_posix())
_ = load_dotenv()

In [ ]:
DATASET_PATH = Path("/data2/training_toolkit_squats/cookbook/squats_v4")

In [ ]:
from training_toolkit import build_trainer, llava_next_video_preset

In [ ]:
from training_toolkit import DataPreset
from training_toolkit.common.video_readers import get_video_reader
import torch
import os
import json


class VideoJSONCollator:
    def __init__(self, processor, num_frames=8, max_length=256):
        self.processor = processor

        self.num_frames = num_frames
        self.max_length = max_length

        self.num_proc = os.cpu_count()
        self.read_video_fn = get_video_reader()

    def __call__(self, examples):
        samples = []
        for example in examples:

            video = torch.tensor(
                self.read_video_fn(
                    example["video"],
                    self.num_frames,
                )
            )

            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "extract JSON."},
                        {"type": "video"},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": json.dumps(example["json"])},
                    ],
                },
            ]

            prompt = self.processor.apply_chat_template(
                conversation, add_generation_prompt=False
            )

            sample = self.processor(
                text=prompt,
                videos=video,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            samples.append(sample)

        padded_inputs = self.processor.tokenizer.pad(
            {
                "input_ids": [
                    sample["input_ids"][0] for sample in samples
                ],  # each element is one batch only so we slice [0]
                "attention_mask": [sample["attention_mask"][0] for sample in samples],
            },
            padding=True,
            return_tensors="pt",
        )

        labels = padded_inputs["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        padded_inputs["labels"] = labels
        padded_inputs["pixel_values_videos"] = torch.cat(
            [sample["pixel_values_videos"] for sample in samples], dim=0
        )
        return padded_inputs


video_json_preset = DataPreset(
    train_test_split=0.2,
    collator_cls=VideoJSONCollator,
)

In [ ]:
training_args = dict(
    **video_json_preset.with_path(DATASET_PATH).as_kwargs(),
    **llava_next_video_preset.as_kwargs()
)
# training_args[...] = ...
# trainer = build_trainer(**training_args)
training_args

In [ ]:
processor = training_args["hf_processor_cls"].from_pretrained(training_args["hf_model_id"])
collator = VideoJSONCollator(processor)

processor.batch_decode(collator([training_args["train_dataset"][0]])["input_ids"])

In [ ]:
trainer = build_trainer(**training_args)

In [ ]:
trainer.train()

In [ ]:
!nvidia-smi